In [ ]:
import pandas as pd

In [ ]:
data = pd.read_csv('https://raw.githubusercontent.com/vkoul/data/main/misc/news_feature.csv')
print(data.shape)
print(data.head())
print(data.info())

(100, 6)
   user_id      group landing_page  time_spent_on_the_page converted  \
0   546592    control          old                    3.48        no   
1   546468  treatment          new                    7.13       yes   
2   546462  treatment          new                    4.40        no   
3   546567    control          old                    3.02        no   
4   546459  treatment          new                    4.75       yes   

  language_preferred  
0            Spanish  
1            English  
2            Spanish  
3             French  
4            Spanish  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 6 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   user_id                 100 non-null    int64  
 1   group                   100 non-null    object 
 2   landing_page            100 non-null    object 
 3   time_spent_on_the_page  100 non-null    float64

In [ ]:
data.groupby('group')['time_spent_on_the_page'].mean()


,time_spent_on_the_page
group,
control,4.5324
treatment,6.2232


In [ ]:
data.groupby('group')['converted'].value_counts()/50


group      converted
control    no           0.58
           yes          0.42
treatment  yes          0.66
           no           0.34
Name: count, dtype: float64

In [ ]:
data.groupby('language_preferred')['converted'].value_counts()/50


language_preferred  converted
English             yes          0.42
                    no           0.22
French              no           0.38
                    yes          0.30
Spanish             yes          0.36
                    no           0.32
Name: count, dtype: float64

In [ ]:
data.groupby('group')['language_preferred'].value_counts()

group      language_preferred
control    French                17
           Spanish               17
           English               16
treatment  French                17
           Spanish               17
           English               16
Name: count, dtype: int64

In [ ]:
data.groupby('landing_page')['language_preferred'].value_counts()

landing_page  language_preferred
new           French                17
              Spanish               17
              English               16
old           French                17
              Spanish               17
              English               16
Name: count, dtype: int64

In [ ]:
#I recommend launching the new landing page. Treatment users spend on average 2 minutes more on the page (6.2 vs 4.5 mins) and convert at a significantly higher rate (66% vs 42%). Language distribution is even across both groups so results are not skewed. English speakers show the highest conversion rate at 42% — worth considering for targeted campaigns.

In [ ]:
import pandas as pd
df = pd.read_csv('https://raw.githubusercontent.com/vkoul/data/main/misc/news_feature.csv')

In [ ]:
df.head()

,user_id,group,landing_page,time_spent_on_the_page,converted,language_preferred
0,546592,control,old,3.48,no,Spanish
1,546468,treatment,new,7.13,yes,English
2,546462,treatment,new,4.40,no,Spanish
3,546567,control,old,3.02,no,French
4,546459,treatment,new,4.75,yes,Spanish


In [ ]:
df.shape


(100, 6)

In [ ]:
df['group'].value_counts()

,count
group,
control,50
treatment,50


In [ ]:

df['landing_page'].value_counts()


,count
landing_page,
old,50
new,50


In [ ]:
df['language_preferred'].value_counts()

,count
language_preferred,
Spanish,34
French,34
English,32


In [ ]:
df['converted'].value_counts()

,count
converted,
yes,54
no,46


# 1 Question : Do users spend more time on the new landing page than old one?

In [ ]:
from scipy import stats

old_page = df[df['landing_page'] == 'old']['time_spent_on_the_page']
new_page = df[df['landing_page'] == 'new']['time_spent_on_the_page']

print(f'Old page mean: {old_page.mean():.2f} minutes')
print(f'New page mean: {new_page.mean():.2f} minutes')

stat, p_value = stats.ttest_ind(new_page, old_page, alternative='greater')
print(f't-statistic: {stat:.4f}')
print(f'p-value: {p_value:.4f}')

Old page mean: 4.53 minutes
New page mean: 6.22 minutes
t-statistic: 3.7868
p-value: 0.0001


Result from this : Users spend significantly more time on the new landing page(6.22 min) than the old page(4.53 min). The difference is larger enough that it's almost certainly real , not random noise

2. Is the conversion rate for the new page greater than the conversion rate for the old page?"

In [ ]:
# Conversion counts by landing page
print(df.groupby('landing_page')['converted'].value_counts())

landing_page  converted
new           yes          33
              no           17
old           no           29
              yes          21
Name: count, dtype: int64


In [ ]:
from statsmodels.stats.proportion import proportions_ztest


conversions = [33, 21]  # new, old


nobs = [50, 50]  # new, old


stat, p_value = proportions_ztest(conversions, nobs, alternative='larger')
print(f'z-statistic: {stat:.4f}')
print(f'p-value: {p_value:.4f}')

z-statistic: 2.4077
p-value: 0.0080


Conclusion:  The new page converts at 66% vs the old page at 42% — a 24-point difference. The two-proportion z-test confirms this difference is statistically significant (p = 0.008), so it's very unlikely to be random chance. The new page genuinely converts better.


3 . Does the converted status depend on the preferred language?

In [ ]:
# Cross-tab of conversion by language
contingency = pd.crosstab(df['language_preferred'], df['converted'])
print(contingency)

converted           no  yes
language_preferred         
English             11   21
French              19   15
Spanish             16   18


In [ ]:
from scipy.stats import chi2_contingency

# Run chi-square test of independence
stat, p_value, dof, expected = chi2_contingency(contingency)

print(f'chi-square statistic: {stat:.4f}')
print(f'p-value: {p_value:.4f}')
print(f'degrees of freedom: {dof}')
print(f'\nExpected counts (if conversion was independent of language):')
print(pd.DataFrame(expected,
                  index=contingency.index,
                  columns=contingency.columns).round(2))

chi-square statistic: 3.0930
p-value: 0.2130
degrees of freedom: 2

Expected counts (if conversion was independent of language):
converted              no    yes
language_preferred              
English             14.72  17.28
French              15.64  18.36
Spanish             15.64  18.36


Conclusion : There is no statistically significant relationship between preferred language and conversion (chi-square = 3.09, p = 0.21). While English speakers appear to convert at a higher rate in this sample, the difference could easily be random noise given the small sample size (~32 users per language). We cannot conclude language affects conversion based on this data."

4 . s the mean time spent on the new page the same for different language users?

In [ ]:

new_page_df = df[df['landing_page'] == 'new']


print(new_page_df.groupby('language_preferred')['time_spent_on_the_page'].mean().round(2))

language_preferred
English    6.66
French     6.20
Spanish    5.84
Name: time_spent_on_the_page, dtype: float64


In [ ]:
from scipy.stats import f_oneway

# Get time spent for each language on the new page
english = new_page_df[new_page_df['language_preferred'] == 'English']['time_spent_on_the_page']
french = new_page_df[new_page_df['language_preferred'] == 'French']['time_spent_on_the_page']
spanish = new_page_df[new_page_df['language_preferred'] == 'Spanish']['time_spent_on_the_page']

stat, p_value = f_oneway(english, french, spanish)
print(f'F-statistic: {stat:.4f}')
print(f'p-value: {p_value:.4f}')

F-statistic: 0.8544
p-value: 0.4320


There is no statistically significant difference in time spent on the new landing page across language groups (F = 0.85, p = 0.43). English, French, and Spanish users spend roughly the same amount of time on the new page